Нам необходимо скорректировать нашу бизнес-стратегию, чтобы соответствовать новым (формирующимся) тенденциям и максимально использовать рыночные возможности.

Нужно ли нам корректировать сегменты клиентов с учётом демографических прогнозов Национального института статистики (INE)?

Необходимо ли адаптировать наши текущие продукты/предложения или создавать новые на основе национальных тенденций в финансовом поведении и уровне финансовых компетенций населения? (ECF)

# Librerias

In [2]:
import pandas as pd
import numpy as np

In [3]:
path_fichero= r"/Users/ekaterinasorokopudova/Desktop/simulador/ProjecteData/Equip_21/Data/ecf_2021.csv"

# EDA técnica

In [4]:
df_fichero = pd.read_csv(
    path_fichero,
    sep=";",
    encoding="utf-8"
)

df_fichero

FileNotFoundError: [Errno 2] No such file or directory: '/Users/ekaterinasorokopudova/Desktop/simulador/ProjecteData/Equip_21/Data/ecf_2021.csv'

In [1]:
df_fichero.head(10)

NameError: name 'df_fichero' is not defined

In [34]:
df_fichero.shape

(7764, 429)

In [35]:
df_fichero.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7764 entries, 0 to 7763
Columns: 429 entries, a01 to ID
dtypes: float64(10), int64(419)
memory usage: 25.4 MB


## % de datos faltantes entre columnas

In [36]:
miss = (
    df_fichero.isna()
      .mean()
      .mul(100)
      .round(2)
      .sort_values(ascending=False)
      .to_frame("missing_pct")
)
display(miss.head())

,missing_pct
a01,0.0
i0700,0.0
i0601,0.0
i0520g,0.0
i0520d,0.0


## Valores únicos por columna 

In [ ]:
nunique = df_fichero.nunique(dropna=True).sort_values()
display(nunique.head()) 
display(nunique.tail())

a01       2
j1400     2
a0900i    2
a0900g    2
b0209     2
dtype: int64

j0700         213
i0700         462
weight       2544
tmp_e0401    5470
ID           7764
dtype: int64

## Datos constantes y casi constantes
- constante - 1 valor único
- casi constante: un modo ocupa >= 99,5% de los valores

In [38]:
const_cols = nunique[nunique <= 1].index.tolist()

In [39]:
def top_share(s: pd.Series) -> float:
    vc = s.value_counts(dropna=True)
    if vc.empty:
        return np.nan
    return vc.iloc[0] / vc.sum()

top_share_series = df_fichero.apply(top_share).sort_values(ascending=False)

near_const_cols = top_share_series[top_share_series >= 0.995].index.tolist()

print("constant cols:", len(const_cols))
print("near-constant cols (>=99.5% same):", len(near_const_cols))

constant cols: 0
near-constant cols (>=99.5% same): 10


### Ejemplos

In [40]:
display(pd.DataFrame({
    "nunique": nunique.loc[near_const_cols],
    "top_share": top_share_series.loc[near_const_cols].round(4)
}).sort_values(["top_share", "nunique"], ascending=[False, True]).head(30))

,nunique,top_share
a0900k,2,0.9997
a0900f,2,0.9990
a0900j,2,0.9986
b0110f,3,0.9979
a0800,15,0.9977
a0900h,2,0.9972
b1207c,4,0.9963
b1207b,4,0.9963
b1207a,4,0.9963
a0900l,2,0.9956


## Comprobación de la validez de las columnas numéricas

In [ ]:
num_cols = df_fichero.select_dtypes(include=["number"]).columns
ranges = df_fichero[num_cols].agg(["min", "max", "mean", "std"]).T
ranges["missing_pct"] = df_fichero[num_cols].isna().mean() * 100
ranges = ranges.round(3).sort_values("std", ascending=False)

display(ranges.head(30))  # los campos más dispersos
display(ranges.tail(30))  # los binarios/casi constantes

,min,max,mean,std,missing_pct
e1400,-99.00,1.100000e+08,14208.355,1248389.044,0.0
e0300,-99.00,1.000000e+08,13157.533,1134903.059,0.0
j0700,-99.00,4.680000e+04,1307.705,3039.705,0.0
weight,420.19,1.941142e+04,4639.769,2990.281,0.0
ID,1.00,7.764000e+03,3882.500,2241.418,0.0
e0500,-99.00,1.700000e+05,253.176,1951.025,0.0
i0800,-99.00,2.022000e+03,1256.460,1003.990,0.0
k0300,-99.00,5.000000e+04,-17.540,941.789,0.0
j0500,-99.00,3.000000e+03,359.509,302.867,0.0
e0800,-99.00,1.250000e+04,90.196,301.189,0.0


,min,max,mean,std,missing_pct
b0202,-97.0,1.0,0.886,1.151,0.0
b0206,-97.0,1.0,0.939,1.132,0.0
j1600,-5.0,3.0,2.590,0.811,0.0
b0110d,-5.0,1.0,0.468,0.525,0.0
b0110a,-5.0,1.0,0.603,0.516,0.0
b0110c,-5.0,1.0,0.373,0.511,0.0
a0000,0.0,1.0,0.519,0.500,0.0
b0110e,-5.0,1.0,0.688,0.492,0.0
a0900b,0.0,1.0,0.622,0.485,0.0
a01,2021.0,2022.0,2021.747,0.435,0.0


In [42]:
profile = pd.DataFrame({
    "dtype": df_fichero.dtypes.astype(str),
    "missing_pct": df_fichero.isna().mean().mul(100).round(2),
    "nunique": df_fichero.nunique(dropna=True),
    "top_share": top_share_series.round(4),
})
profile = profile.sort_values(["missing_pct", "top_share"], ascending=[False, False])
display(profile.head(40))

,dtype,missing_pct,nunique,top_share
a0900k,int64,0.0,2,0.9997
a0900f,int64,0.0,2,0.9990
a0900j,int64,0.0,2,0.9986
b0110f,int64,0.0,3,0.9979
a0800,int64,0.0,15,0.9977
a0900h,int64,0.0,2,0.9972
b1207a,int64,0.0,4,0.9963
b1207b,int64,0.0,4,0.9963
b1207c,int64,0.0,4,0.9963
a0900l,int64,0.0,2,0.9956
